# Part 2: Preprocessing

In [34]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

## Part 2.1: Preprocessing

In [35]:
# Load dataset
train = pd.read_csv(r"C:\Users\alsuw\Desktop\MLSD project\data\train.csv")

train = train.drop(columns=["Unnamed: 0"])

# Log transform large numerical columns
log_cols = ["#follows", "#followers", "description length", "#posts"]
for col in log_cols:
    train[col] = np.log1p(train[col])

In [36]:
categorical_cols = ["profile pic", "name==username", "external URL", "private"]
categorical_cols = [col for col in categorical_cols if col in train.columns]

numeric_columns = [col for col in train.columns if col not in categorical_cols and col != "fake"]


In [37]:
# Split features and target
X = train.drop(columns=["fake"])
y = train["fake"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=10,)


In [38]:
# Preprocessing pipelines
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)


In [39]:
# Apply preprocessing
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)



In [40]:
# Get transformed feature names
num_features = numeric_columns
cat_features = preprocessor.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_cols)

all_features = np.concatenate([num_features, cat_features])


In [41]:
# Convert to DataFrames
train_processed = pd.DataFrame(X_train_processed, columns=all_features)
val_processed = pd.DataFrame(X_val_processed, columns=all_features)

# Add target back
train_processed["fake"] = y_train.reset_index(drop=True)
val_processed["fake"] = y_val.reset_index(drop=True)


In [42]:
# Save as parquet
train_processed.to_parquet(r"C:\Users\alsuw\Desktop\MLSD project\data\preprocessed\train_processed.parquet", index=False)

val_processed.to_parquet(r"C:\Users\alsuw\Desktop\MLSD project\data\preprocessed\val_processed.parquet", index=False)

print("Done")

Done


In [43]:
train_check = pd.read_parquet(r"C:\Users\alsuw\Desktop\MLSD project\data\preprocessed\train_processed.parquet")

val_check = pd.read_parquet(r"C:\Users\alsuw\Desktop\MLSD project\data\preprocessed\val_processed.parquet")

print("Train:")
print(train_check.head(0))

print("\nValidation:")
print(val_check.head(0))

Train:
Empty DataFrame
Columns: [nums/length username, fullname words, nums/length fullname, description length, #posts, #followers, #follows, profile pic_0.0, profile pic_1.0, name==username_0.0, name==username_1.0, external URL_0.0, external URL_1.0, private_0.0, private_1.0, fake]
Index: []

Validation:
Empty DataFrame
Columns: [nums/length username, fullname words, nums/length fullname, description length, #posts, #followers, #follows, profile pic_0.0, profile pic_1.0, name==username_0.0, name==username_1.0, external URL_0.0, external URL_1.0, private_0.0, private_1.0, fake]
Index: []


In [44]:
print(train_processed.shape)
print(train_processed.head())

(2000, 16)
   nums/length username  fullname words  nums/length fullname  \
0              -0.58272       -0.353967             -0.259167   
1              -0.58272       -0.353967             -0.259167   
2              -0.58272       -0.353967             -0.259167   
3              -0.58272        0.589944             -0.259167   
4              -0.58272       -0.353967             -0.259167   

   description length    #posts  #followers  #follows  profile pic_0.0  \
0           -0.578976 -0.760238   -0.359601 -0.232885              0.0   
1            0.706048  0.315246   -0.661133 -0.321973              0.0   
2            0.212312 -0.699321   -0.622725 -0.560782              0.0   
3            0.310403  1.139500    1.296358  0.599260              0.0   
4           -0.974620 -0.909027   -0.364043 -0.268624              0.0   

   profile pic_1.0  name==username_0.0  name==username_1.0  external URL_0.0  \
0              1.0                 1.0                 0.0               

# Part 3

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np

In [46]:
# Logistic Regression model evaluation function
def evaluate_model(X, y, name):
    model = LogisticRegression(max_iter=1000)
    scores = cross_validate(model, X, y, cv=5, scoring=["accuracy", "precision", "recall", "f1"])
    print(f"{name}")
    print(f"F1: {np.mean(scores['test_f1']):.4f}, Precision: {np.mean(scores['test_precision']):.4f}, Recall: {np.mean(scores['test_recall']):.4f}") 

In [47]:
#Feature Selection and Model Evaluation 
target_col = "fake"

X_all = train_processed.drop(columns=[target_col])
y = train_processed[target_col]

important_features = ["#posts", "#followers", "#follows", "description length"]

X_selected = train_processed[important_features]

evaluate_model(X_all, y, "\nAll Features")

evaluate_model(X_selected, y, "\nImportant Features")

X_weighted = X_all.copy()
for col in important_features:
    X_weighted[col] = X_weighted[col] * 2
evaluate_model(X_weighted, y, "\nWeighted Important Features")


# Try different C values
for c in [0.1, 0.5, 1, 5, 10]:
    model = LogisticRegression(C=c, max_iter=1000)
    
    scores = cross_validate(model, X_all, y, cv=5, scoring=["f1", "precision", "recall"])
    
    print(f"\nC = {c}")
    print("F1 Score  :", scores["test_f1"].mean())
    print("Precision :", scores["test_precision"].mean())
    print("Recall    :", scores["test_recall"].mean())





All Features
F1: 0.9549, Precision: 0.9818, Recall: 0.9300

Important Features
F1: 0.8411, Precision: 0.8943, Recall: 0.7950

Weighted Important Features
F1: 0.9589, Precision: 0.9820, Recall: 0.9375

C = 0.1
F1 Score  : 0.9394633937068834
Precision : 0.9862454671831481
Recall    : 0.8975

C = 0.5
F1 Score  : 0.949495481298887
Precision : 0.9816482145447537
Recall    : 0.9199999999999999

C = 1
F1 Score  : 0.954908853791394
Precision : 0.9817915664004344
Recall    : 0.93

C = 5
F1 Score  : 0.9603853355532415
Precision : 0.9795283531366141
Recall    : 0.9425000000000001

C = 10
F1 Score  : 0.9603853355532415
Precision : 0.9795283531366141
Recall    : 0.9425000000000001


In [48]:
# Add features

X_new = train_processed.copy()

# New features
X_new["engagement_ratio"] = X_new["#followers"] / (X_new["#follows"] + 1e-6)
X_new["activity_rate"] = X_new["#posts"] / (X_new["#followers"] + 1e-6)
X_new["follow_ratio"] = X_new["#follows"] / (X_new["#followers"] + 1e-6)

# Clean any bad values
X_new = X_new.replace([np.inf, -np.inf], 0)
X_new = X_new.fillna(0)


# Important + new only
selected_plus_new = important_features + [
    "engagement_ratio",
    "activity_rate",
    "follow_ratio"
]

evaluate_model(X_new[selected_plus_new], y_new, "\nImportant + New Features Only")



Important + New Features Only
F1: 0.8379, Precision: 0.8863, Recall: 0.7950


### C5 or C10 seems to give the best balance of precision and recall.